In [1]:
from common import *
from tqdm import tqdm
import numpy as np
import os

/mnt/c/Users/Usuario UTP/Documents/tareas/redNeuronal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from socket import socket
from typing import Dict, Any
import pickle
import json
import shutil
import os

class State:
    
    def __init__(self, params: Dict[str, Any]):
        self.data = params
    
    def do(self, sock: socket):
        self.data["status"] = "ok"
    
    def next(self):
        if self.data.get("status") == "error":
            return None
        return HandShake(self.data)

class HandShake(State):
    
    def do(self, sock: socket):
        super().do(sock)
        try:
            dataLen = recvall(sock, 8)
            data = recvall(sock, int.from_bytes(dataLen, 'big')).decode("utf-8")
            self.data.update(json.loads(data))
            self.data["shape"] = tuple(self.data["shape"])
            seed = int(self.data["seed"])
            os.environ["token"] = self.data["token"]
            self.data["sequential"] = Sequential.load(self.data["sequential"])
            match self.data["error"]:
                case "mse":
                    self.data["error"] = mse
                case "lostEntropy":
                    self.data["error"] = lostEntropy
            match self.data["devError"]:
                case "devMse":
                    self.data["devError"] = devMse
                case "devLostEntropy":
                    self.data["devError"] = devLostEntropy
        except Exception as e:
            print("HandShake error: ", e)
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        return Recolection(self.data)
    
class Recolection(State):
    
    def do(self, sock: socket):
        super().do(sock)
        shardPosition = self.data.get("shardPosition")
        try:
            dsName = self.data["dsName"]
            split = self.data["split"]
            (ds, size) = downloadDataset(dsName, split)
            self.data["ds"] = ds
            self.data["dsSize"] = size
            self.data["shard"] = int(size/(self.data["batchSize"]*self.data["workers"]))
            self.data["dsSize"] = self.data["datasetPorcent"]*self.data["dsSize"] / self.data["workers"]
            if self.data["test"]:
                (dsTest, dsSizeTest) = downloadDataset(dsName, "test")
                self.data["xTest"], self.data["yTest"] = next(
                    getBatch(
                        dsTest, 
                        dsSizeTest, 
                        self.data["labels"], 
                        dsSizeTest, 
                        self.data["workers"], 
                        (self.data["shard"], self.data.get("shardPosition")), 
                        shape=self.data["shape"], 
                        tqdmDisable=False,
                        classNumber=self.data["labelsNumber"]
                    ))
            message = f"y-{shardPosition}"
            sock.sendall(message.encode("utf-8"))
        except Exception as e:
            print("Recolection error: ", e)
            message = f"n-{shardPosition}"
            try:
                sock.sendall(message.encode("utf-8"))
            except:
                pass
            sock.close()
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        return TrainBatch(self.data)

class TrainBatch(State):
    
    def do(self, sock: socket):
        super().do(sock)
        shardPosition = self.data.get("shardPosition")
        try:
            workers = self.data["workers"]
            length_prefix = recvall(sock, 8)
            message_length = int.from_bytes(length_prefix, 'big')
            data = recvall(sock, message_length)
            self.data["w"], self.data["b"] = pickle.loads(data)
            ds = self.data["ds"]
            batchSize = self.data["batchSize"]
            labels = self.data["labels"]
            size = self.data["dsSize"]
            w_grad_batch = [np.zeros_like(wi) for wi in self.data["w"]]
            b_grad_batch = [np.zeros_like(bi) for bi in self.data["b"]]
            sequential = self.data["sequential"]
            devError = self.data["devError"]
            split = self.data["split"]
            shard = self.data["shard"]
            if "batch_gen" not in self.data:
                self.data["batch_gen"] = getBatch(
                            ds, 
                            batchSize, 
                            labels, 
                            size, 
                            workers, 
                            (shard, shardPosition), 
                            shape=self.data["shape"], 
                            tqdmDisable=False,
                            classNumber=self.data["labelsNumber"]
                        )
            try:
                self.data["x"], self.data["y"] = next(self.data["batch_gen"])
                for x_b, y_b in zip(self.data["x"], self.data["y"]):
                    batch(x_b, y_b, self.data["w"], self.data["b"], w_grad_batch, b_grad_batch, sequential, devError)
            except StopIteration:
                del self.data["batch_gen"]
                
            self.data["w_grad_batch"] = w_grad_batch
            self.data["b_grad_batch"] = b_grad_batch
        except Exception as e:
            print("TrainBatch error: ", e)
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        if self.data["verbose"]:
            return Validate(self.data)
        return End(self.data)

class Validate(State):
    
    def do(self, sock: socket):
        super().do(sock) # No olvides inicializar el estado como "ok"
        try:
            (accuracy, batch_losses) = evaluate(
                self.data["w"], 
                self.data["b"], 
                self.data["x"], 
                self.data["y"], 
                self.data["sequential"], 
                self.data["error"]
            )
            test_accuracy = 0
            if self.data.get("test"):
                shard = self.data["shard"]
                shardPosition = self.data.get("shardPosition")
                (test_accuracy, _) = evaluate(
                    self.data["w"], 
                    self.data["b"], 
                    self.data["xTest"], 
                    self.data["yTest"], 
                    self.data["sequential"], 
                    self.data["error"]
                )
            datos_metricas = pickle.dumps((accuracy, test_accuracy, batch_losses))
            sock.sendall(len(datos_metricas).to_bytes(8, 'big'))
            sock.sendall(datos_metricas)
            
        except Exception as e:
            print("Validate error: ", e)
            self.data["status"] = "error"
    
    def next(self):
        if self.data.get("status") == "error":
            return None
        return End(self.data)

class End(State):
    
    def do(self, sock: socket):
        super().do(sock)
        shardPosition = self.data.get("shardPosition")
        try:
            message = f"y-{shardPosition}"
            sock.sendall(message.encode("utf-8"))
            w_grad_batch = self.data.get("w_grad_batch", [])
            b_grad_batch = self.data.get("b_grad_batch", [])
            data_to_send = pickle.dumps((w_grad_batch, b_grad_batch))
            sock.sendall(len(data_to_send).to_bytes(8, 'big'))
            sock.sendall(data_to_send)
            continueWork = recvall(sock, 1).decode("utf-8")
            self.data["continue"] = continueWork
        except Exception as e:
            print("End error: ", e)
            self.data["status"] = "error"
            
    def next(self):
        if self.data.get("status") == "error":
            return None
        match self.data.get("continue"):
            case "y":
                return TrainBatch(self.data)
            case "n":
                for folder in os.listdir('.'):
                    if folder.startswith('data-') and os.path.isdir(folder):
                        shutil.rmtree(folder)
                return None
            case _:
                return None


In [4]:
def __fit(host, port):
    import socket
    estado_actual = State({})
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.connect((host, port))
            print(f"Conectado a {host}:{port}")
            while estado_actual is not None:
                estado_actual.do(s)
                estado_actual = estado_actual.next()
        except Exception as e:
            print(e)
            s.close()

In [13]:
HOST = "127.0.0.1"
PORT = 65432
__fit(HOST, PORT)

Conectado a 127.0.0.1:65432


batch:   0%|          | 0/1 [01:15<?, ?it/s]
